In [1]:
# Cell 1: Install dependencies
!pip install -q transformers datasets accelerate evaluate librosa soundfile sentencepiece scikit-learn pythainlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 109.7 MB/s eta 0:00:00


In [2]:
# Cell 2: Mount Drive + paths config
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE     = '/content/drive/MyDrive/ThaiScamCall'
ZIP_PATH       = f'{DRIVE_BASE}/mp3_15s_clean.zip'
EXTRACT_DIR    = '/content/data_15s_clean'
SPLITS_DIR     = f'{DRIVE_BASE}/splits'
TRANSCRIPTS_DIR = f'{DRIVE_BASE}/transcripts'
RUNS_DIR       = f'{DRIVE_BASE}/runs/text_cls'
os.makedirs(TRANSCRIPTS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

Mounted at /content/drive


In [3]:
# Cell 3: Extract zip + load splits
import zipfile
if not os.path.exists(EXTRACT_DIR) or len(os.listdir(EXTRACT_DIR)) < 1000:
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
print('Files:', len(os.listdir(EXTRACT_DIR)))

import pandas as pd
splits = {s: pd.read_csv(f'{SPLITS_DIR}/{s}.csv') for s in ['train', 'val', 'test']}
for s, d in splits.items():
    print(f'{s:5s} n={len(d):>6} | {d.label.value_counts().to_dict()}')

Files: 19591
train n= 15672 | {1: 7980, 0: 7692}
val   n=  1959 | {1: 998, 0: 961}
test  n=  1960 | {1: 998, 0: 962}


In [4]:
# Cell 4: ASR models config (Typhoon disabled — HF processor incompatible)
ASR_MODELS = {
    'thonburian': 'biodatlab/whisper-th-medium-combined',
    #'typhoon': 'scb10x/typhoon-asr-realtime',  # disabled
}

In [ ]:
# Cell 5: ASR transcription (direct model+processor + BATCHED + suppress warnings)
# ⚠️ ถ้า transcripts มีใน Drive แล้ว → ข้าม cell นี้ได้
import torch, gc, librosa, warnings, logging
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, AutoModelForCTC
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

SAVE_EVERY_BATCHES = 25
BATCH = 8
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32

def transcribe_with(asr_name, model_id):
    out_dir = f'{TRANSCRIPTS_DIR}/{asr_name}'
    os.makedirs(out_dir, exist_ok=True)

    processor = AutoProcessor.from_pretrained(model_id)
    try:
        model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, torch_dtype=dtype).to(device)
        is_seq2seq = True
    except Exception:
        model = AutoModelForCTC.from_pretrained(model_id, torch_dtype=dtype).to(device)
        is_seq2seq = False
    model.eval()
    print(f'[{asr_name}] loaded ({"seq2seq" if is_seq2seq else "ctc"})')

    forced_ids = None
    if is_seq2seq and 'whisper' in model_id.lower():
        forced_ids = processor.get_decoder_prompt_ids(language='th', task='transcribe')

    def transcribe_batch(paths):
        audios = [librosa.load(p, sr=16000)[0] for p in paths]
        inputs = processor(audios, sampling_rate=16000, return_tensors='pt', padding=True)
        inputs = {k: (v.to(device, dtype=dtype) if v.dtype == torch.float32 else v.to(device))
                  for k, v in inputs.items()}
        with torch.no_grad():
            if is_seq2seq:
                gen = model.generate(**inputs, forced_decoder_ids=forced_ids)
                return processor.batch_decode(gen, skip_special_tokens=True)
            else:
                logits = model(**inputs).logits
                pred_ids = logits.argmax(dim=-1)
                return processor.batch_decode(pred_ids, skip_special_tokens=True)

    # sanity check
    test_file = f'{EXTRACT_DIR}/{splits["val"].iloc[0].filename}'
    sanity = transcribe_batch([test_file])[0]
    print(f'[sanity] {sanity[:100]!r}')
    assert sanity.strip(), 'sanity failed!'

    for split in ['val', 'test', 'train']:
        df = splits[split].reset_index(drop=True)
        out_path = f'{out_dir}/{split}.csv'

        done_map = {}
        if os.path.exists(out_path):
            prev = pd.read_csv(out_path)
            done_map = {r.filename: r.text for _, r in prev.iterrows()
                        if isinstance(r.text, str) and r.text}
            print(f'[{asr_name}/{split}] resume: {len(done_map)}/{len(df)} done')

        rows, todo = [], []
        for _, r in df.iterrows():
            if r.filename in done_map:
                rows.append({'filename': r.filename, 'label': r.label, 'text': done_map[r.filename]})
            else:
                rows.append({'filename': r.filename, 'label': r.label, 'text': None})
                todo.append(len(rows) - 1)

        err_count = 0
        for n_batch, b_start in enumerate(tqdm(range(0, len(todo), BATCH),
                                                desc=f'{asr_name}/{split}')):
            batch = todo[b_start:b_start + BATCH]
            paths = [f'{EXTRACT_DIR}/{rows[i]["filename"]}' for i in batch]
            try:
                texts = transcribe_batch(paths)
                for idx, text in zip(batch, texts):
                    rows[idx]['text'] = text
            except Exception as e:
                err_count += 1
                if err_count <= 3:
                    print(f'batch err: {str(e)[:80]}')
                for idx in batch:
                    try:
                        rows[idx]['text'] = transcribe_batch(
                            [f'{EXTRACT_DIR}/{rows[idx]["filename"]}'])[0]
                    except Exception:
                        rows[idx]['text'] = ''

            if (n_batch + 1) % SAVE_EVERY_BATCHES == 0:
                pd.DataFrame(rows).to_csv(out_path, index=False)

        pd.DataFrame(rows).to_csv(out_path, index=False)
        print(f'[{asr_name}/{split}] saved (batch errs: {err_count})')

    del model, processor
    gc.collect()
    torch.cuda.empty_cache()

for name, mid in ASR_MODELS.items():
    transcribe_with(name, mid)

preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/948 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.75k [00:00<?, ?B/s]

[thonburian] loaded (seq2seq)
[sanity] 'หวัดดีครับ มึงอยู่ไหนอ่ะ อ้าว เฮ้ย กูอยู่บ้านทำไม อยากไปดูหนังกันไหม มีหนังใหม่เข้ามาแล้วนะ อืม ก็เอ'
[thonburian/val] resume: 1949/1959 done


thonburian/val:   0%|          | 0/224 [00:00<?, ?it/s]

In [ ]:
# Cell 6: TF-IDF + LR baseline (text classifier ต้องชนะ)
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from pythainlp.tokenize import word_tokenize

BASELINE_DIR = f'{DRIVE_BASE}/runs/baselines'
os.makedirs(BASELINE_DIR, exist_ok=True)

def th_tok(s):
    return [w for w in word_tokenize(s, engine='newmm') if w.strip()]

def save_baseline_preds(name, test_df, y_pred, y_score):
    out_dir = f'{BASELINE_DIR}/{name}'
    os.makedirs(out_dir, exist_ok=True)
    pd.DataFrame({
        'filename':  test_df.filename.values,
        'label':     test_df.label.values,
        'pred':      y_pred,
        'prob_scam': y_score,
    }).to_csv(f'{out_dir}/test_predictions.csv', index=False)
    y = test_df.label.values
    print(f'[{name}] acc={accuracy_score(y, y_pred):.4f}  '
          f'macro_f1={f1_score(y, y_pred, average="macro"):.4f}  '
          f'auc={roc_auc_score(y, y_score):.4f}')

# loop ทุก ASR ที่ถอดเสียงไว้
for asr_name in ASR_MODELS.keys():
    tr_train = pd.read_csv(f'{TRANSCRIPTS_DIR}/{asr_name}/train.csv')
    tr_test  = pd.read_csv(f'{TRANSCRIPTS_DIR}/{asr_name}/test.csv')
    tr_train['text'] = tr_train['text'].fillna('').astype(str)
    tr_test['text']  = tr_test['text'].fillna('').astype(str)

    # align test order ให้ตรงกับ splits['test']
    test_aligned = splits['test'][['filename', 'label']].merge(
        tr_test[['filename', 'text']], on='filename', how='left')
    test_aligned['text'] = test_aligned['text'].fillna('')

    vec = TfidfVectorizer(tokenizer=th_tok, max_features=20000,
                          ngram_range=(1, 2), min_df=2)
    Xtr = vec.fit_transform(tr_train.text)
    Xte = vec.transform(test_aligned.text)

    clf = LogisticRegression(max_iter=2000, C=1.0, n_jobs=-1).fit(Xtr, tr_train.label.values)
    save_baseline_preds(f'tfidf_lr_{asr_name}', test_aligned,
                        clf.predict(Xte), clf.predict_proba(Xte)[:, 1])

In [ ]:
# Cell 8: Text classifier models config (WangchanBERTa disabled — CamemBERT tokenizer version conflict)
TEXT_MODELS = {
    # 'wangchan':  'airesearch/wangchanberta-base-att-spm-uncased',  # disabled
    'phayathai': 'clicknext/phayathaibert',
}

In [ ]:
# Cell 9: Fine-tune text classifier (PhayaThaiBERT only, transformers v5 compat)
import numpy as np, evaluate
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, DataCollatorWithPadding)
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

f1_m  = evaluate.load('f1')
acc_m = evaluate.load('accuracy')
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        **acc_m.compute(predictions=preds, references=p.label_ids),
        **f1_m.compute(predictions=preds, references=p.label_ids, average='macro'),
    }

def train_combo(asr_name, text_name, text_model_id):
    combo = f'{asr_name}_{text_name}'
    out_dir = f'{RUNS_DIR}/{combo}'
    os.makedirs(out_dir, exist_ok=True)
    print(f'\n========== {combo} ==========')

    tok = AutoTokenizer.from_pretrained(text_model_id)

    def load_ds(split):
        df = pd.read_csv(f'{TRANSCRIPTS_DIR}/{asr_name}/{split}.csv')
        df['text'] = df['text'].fillna('').astype(str)
        ds = Dataset.from_pandas(df[['filename', 'text', 'label']])
        return ds.map(lambda x: tok(x['text'], truncation=True, max_length=256), batched=True)

    ds = {s: load_ds(s) for s in ['train', 'val', 'test']}
    model = AutoModelForSequenceClassification.from_pretrained(text_model_id, num_labels=2)

    args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=3,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        learning_rate=2e-5,
        warmup_steps=200,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        fp16=True,
        logging_steps=50,
        save_total_limit=1,
        report_to='none',
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=ds['train'], eval_dataset=ds['val'],
        processing_class=tok,
        data_collator=DataCollatorWithPadding(tok),
        compute_metrics=compute_metrics,
    )
    trainer.train()

    pred = trainer.predict(ds['test'])
    y_pred  = np.argmax(pred.predictions, axis=1)
    y_true  = pred.label_ids
    y_score = torch.softmax(torch.tensor(pred.predictions), dim=1)[:, 1].numpy()

    test_df = pd.read_csv(f'{TRANSCRIPTS_DIR}/{asr_name}/test.csv')
    out = pd.DataFrame({
        'filename': test_df.filename,
        'label':    y_true,
        'pred':     y_pred,
        'prob_scam': y_score,
    })
    pred_path = f'{out_dir}/test_predictions.csv'
    out.to_csv(pred_path, index=False)

    print(classification_report(y_true, y_pred, target_names=['not_scam', 'scam'], digits=4))
    print('Confusion:\n', confusion_matrix(y_true, y_pred))
    print('ROC-AUC:', roc_auc_score(y_true, y_score))
    print('saved ->', pred_path)

    del trainer, model, tok, ds
    gc.collect()
    torch.cuda.empty_cache()

# Loop: thonburian × PhayaThaiBERT (1 combo)
for text_name, text_id in TEXT_MODELS.items():
    train_combo('thonburian', text_name, text_id)